In [1]:
import pandas as pd
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random
import torch.autograd as autograd
import matplotlib.pyplot as plt
import time
import math
from torch.optim.lr_scheduler import StepLR
from scipy.optimize import minimize
from scipy.stats import norm
import warnings  
import statsmodels.api as smapi 
from statsmodels.genmod.families import Poisson
from sklearn.utils import resample
import os
import re
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
torch.manual_seed(25)

In [2]:
#define Neural Network for propensity score
class PS(nn.Module):
    def __init__(self, in_N, m, depth=2):
        super(PS, self).__init__()
        self.stack = nn.ModuleList() 
        self.stack.append(nn.Linear(in_N, m))
        for i in range(depth-1):
            self.stack.append(nn.Linear(m, m))
        self.stack.append(nn.Linear(m, 1))
        self.sigmoid = nn.Sigmoid()
        self.act = nn.ReLU() 
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = self.act(self.stack[0](x))
        x = self.dropout(x)   
        for i in range(1,len(self.stack)-1):
            x =  self.act(self.stack[i](x))   
            x = self.dropout(x)   
        x = self.stack[-1](x)
        x = 1e-2+ (0.98)*self.sigmoid(x)       #0.01-0.99
        return  x
    
def weights_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_normal_(m.weight)
        nn.init.constant_(m.bias, 0.0)


#define Neural Network for prune
class M_pruned(nn.Module):
    def __init__(self, in_N, m, depth=2):
        super(M_pruned, self).__init__()
         
        self.stack = nn.ModuleList() 
        self.stack.append(nn.Linear(in_N, m)) 
        for i in range(depth-1):
            self.stack.append(nn.Linear(m, m)) 
        self.stack.append(nn.Linear(m, 1))
        self.sigmoid = nn.Sigmoid()
        self.act = nn.ReLU()
        # Dropout layer 
        self.dropout = nn.Dropout(0.1)
         

    def forward(self, x):
        x = self.act(self.stack[0](x))
        x = self.dropout(x)
        for i in range(1,len(self.stack)-1):
            x =  self.act(self.stack[i](x))    
            x = self.dropout(x)
        x = self.stack[-1](x)
        x = 1e-2+ (0.98)*self.sigmoid(x)  
        return  x
    
    def pre_act(self, x):
        xi = self.stack[0](x)
        pre_act_list = [xi]
        x = self.act(xi)
        x = self.dropout(x)
        for i in range(1,len(self.stack)-1):
            xi = self.stack[i](x)
            pre_act_list.append(xi)
            x =  self.act(xi)  
            x = self.dropout(x)
        x = self.stack[-1](x)
        x = 1e-2+ (0.98)*self.sigmoid(x)   
        return  x, pre_act_list

     

def test_NCO(nco, Xn, Wn):
    NCO_pred = nco.predict_proba(Xn)[:,1] 
    NCO_pred_binary = (NCO_pred >= 0.5) 
    act = Wn.squeeze() 
    accuracy = np.sum(NCO_pred_binary == act )/act.shape[0] 
    return NCO_pred

def RR(weights,Ar, Yr):
    data = pd.DataFrame({'Y': Yr, 'A': Ar, 'weights': weights})
    AA = smapi.add_constant(data['A'])   
    poisson_model = smapi.GLM(data['Y'], AA, family=smapi.families.Poisson(), freq_weights=data['weights']).fit()
    logRR = poisson_model.params['A'] 
    selogrr =  poisson_model.bse['A'] 
    return  logRR , selogrr

 

### Load simulated data

In [ ]:
 
df_all = pd.read_csv('simulated_data.csv', on_bad_lines='skip') 
rows, cols = df_all.shape
print(f"Number of rows: {rows}")
print(f"Number of columns: {cols}")
column_names_list = list(df_all.columns)

 

Number of rows: 50000
Number of columns: 152


In [5]:
column_names_list = list(df_all.columns)
for i in range(df_all.shape[1]):
    print(i, column_names_list[i])


0 covariate_1
1 covariate_2
2 covariate_3
3 covariate_4
4 covariate_5
5 covariate_6
6 covariate_7
7 covariate_8
8 covariate_9
9 covariate_10
10 covariate_11
11 covariate_12
12 covariate_13
13 covariate_14
14 covariate_15
15 covariate_16
16 covariate_17
17 covariate_18
18 covariate_19
19 covariate_20
20 covariate_21
21 covariate_22
22 covariate_23
23 covariate_24
24 covariate_25
25 covariate_26
26 covariate_27
27 covariate_28
28 covariate_29
29 covariate_30
30 covariate_31
31 covariate_32
32 covariate_33
33 covariate_34
34 covariate_35
35 covariate_36
36 covariate_37
37 covariate_38
38 covariate_39
39 covariate_40
40 covariate_41
41 covariate_42
42 covariate_43
43 covariate_44
44 covariate_45
45 covariate_46
46 covariate_47
47 covariate_48
48 covariate_49
49 covariate_50
50 covariate_51
51 covariate_52
52 covariate_53
53 covariate_54
54 covariate_55
55 covariate_56
56 covariate_57
57 covariate_58
58 covariate_59
59 covariate_60
60 covariate_61
61 covariate_62
62 covariate_63
63 covariat

### Step 1. train a neural network to estimate propensity scores from baseline covariates

In [7]:
  
df = df_all.copy()
Y =  df['outcome'].to_numpy()  
A =  df['treatment'].to_numpy()   
x_df = df.filter(regex='covariate_') 
X = x_df.astype(np.float32).to_numpy()  
Xt = torch.tensor(X, dtype=torch.float32)
At = torch.tensor(A.reshape(-1, 1), dtype=torch.float32)   

 

### PS prediction
d = X.shape[1]
n_hidden = 300
n_depth = 3
model = PS(in_N=d , m= n_hidden, depth= n_depth)   
model.apply(weights_init)
print(model) 

criterion = nn.BCELoss()  
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-6)
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)    
n_epochs = 25
batch_size = 80   
#
model.eval()   
with torch.no_grad():   
    test_outputs = model(Xt).squeeze()   
    predicted_prob = test_outputs.numpy()
actual_treatment = At.numpy().squeeze()
pred = (predicted_prob >=0.5)
print('initialzation', np.sum(pred==actual_treatment )/pred.shape[0],optimizer.param_groups[-1]['lr'])
#
for epoch in range(n_epochs):  
    model.train()  
    running_loss = 0.0
    permutation = torch.randperm(Xt.size(0)) 
    for i in range(0, Xt.shape[0], batch_size):
        ind = permutation[i:i + batch_size]
        batch_x, batch_a = Xt[ind], At[ind]
        out = model(batch_x)  
        optimizer.zero_grad()
        loss = criterion(out,batch_a)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()

    model.eval()   
    with torch.no_grad():   
        test_outputs = model(Xt).squeeze()   
        predicted_prob = test_outputs.numpy()
    actual_treatment = At.numpy().squeeze()
    pred = (predicted_prob >=0.5) 
    if (epoch + 1) %1 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}, Loss: {running_loss/len(Xt)*batch_size}', np.sum(pred==actual_treatment )/pred.shape[0],optimizer.param_groups[-1]['lr'])
    if np.sum(pred==actual_treatment )/pred.shape[0]>0.95:
        break

PS(
  (stack): ModuleList(
    (0): Linear(in_features=100, out_features=300, bias=True)
    (1-2): 2 x Linear(in_features=300, out_features=300, bias=True)
    (3): Linear(in_features=300, out_features=1, bias=True)
  )
  (sigmoid): Sigmoid()
  (act): ReLU()
  (dropout): Dropout(p=0.1, inplace=False)
)
initialzation 0.54146 0.001
Epoch 1/25, Loss: 0.6781666764259338 0.59298 0.001
Epoch 2/25, Loss: 0.6764541870117187 0.59298 0.001
Epoch 3/25, Loss: 0.6759951618194581 0.59298 0.001
Epoch 4/25, Loss: 0.6758305577278136 0.59298 0.001
Epoch 5/25, Loss: 0.6750421047210693 0.59298 0.001
Epoch 6/25, Loss: 0.6740905246734619 0.59298 0.001
Epoch 7/25, Loss: 0.6721767121315002 0.593 0.001
Epoch 8/25, Loss: 0.6700443088531494 0.59368 0.001
Epoch 9/25, Loss: 0.6666059223175048 0.59708 0.001
Epoch 10/25, Loss: 0.6620604035377503 0.60734 0.0005
Epoch 11/25, Loss: 0.6471934106826782 0.64038 0.0005
Epoch 12/25, Loss: 0.6385929472923279 0.65472 0.0005
Epoch 13/25, Loss: 0.6278951192855835 0.66704 0.000

### Step 2. compute CATEs on multiple NCOs to stratify individuals into Biased and Reliable groups

In [8]:
 
### NCO prediction
NCO = []
j = 0
for i in range(100,149+1):
    j += 1
    if np.mean(df[column_names_list[i]].to_numpy())>= 0.001:
        NCO.append(i)
print(len(NCO ))


#use poisson regression for each W on A:  using propensity score
model.eval()   
with torch.no_grad():
    ps = model(torch.tensor(X, dtype=torch.float32)).numpy().squeeze()   
wt = np.ones(ps.shape[0])
wt[A==1] = 1/ps[A==1]
wt[A==0] = 1/(1-ps[A==0])
 
W_all = NCO
print(len(W_all), W_all)
index_split = random.sample(range(len(W_all)), math.ceil( len(W_all)/3 )) 
W_split = [W_all[i] for i in index_split]
print(len(W_split ), W_split )

    
#
nco_dict = {}
NCO_select = W_split

j = 0
for i in NCO_select:  
    W =  df[column_names_list[i]].to_numpy()  
    #treatment 1
    lr1 = LogisticRegression(max_iter=1000, solver='liblinear')
    lr1.fit(  X[A==1 ],W[A==1] ) 
    nco1 = lr1.predict_proba(X)[:,1]
        
    #treatment 0
    lr0 = LogisticRegression(max_iter=1000, solver='liblinear')
    lr0.fit(  X[A==0 ],W[A==0] ) 
    nco0 = lr1.predict_proba(X)[:,1]

    NCO1_pred = test_NCO(lr1, X , W )
    NCO0_pred = test_NCO(lr0, X , W ) 

    #compare
    nco_dict[str(i)+'nco1_p'] = NCO1_pred
    nco_dict[str(i)+'nco0_p'] = NCO0_pred
    j += 1
 
#unweighted median
for i in NCO_select:  
    W =  df[column_names_list[i]].to_numpy()  
    NCO_pred0  =   nco_dict[str(i)+'nco0_p'] 
    NCO_pred1  =  nco_dict[str(i)+'nco1_p'] 
    ind = (abs(NCO_pred0-NCO_pred1)>0.05).reshape(-1,1)   
    if i == NCO_select[0]: 
        I = ind
    else:
        I = np.hstack((I,ind))
print(I.shape)
avg_I = np.sum(I,axis=1)  
print(np.max(avg_I))
con = (avg_I>=(np.max(avg_I)//2     ) )  
X_h = Xt[con]
X_l = Xt[~con]
A_h = At[con]
A_l = At[~con]
print(X_h.shape, X_l.shape,A_h.shape, A_l.shape)
print('high biased ratio: ', X_h.shape[0]/avg_I.shape[0])
 


50
50 [100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149]
17 [123, 101, 128, 107, 112, 110, 119, 147, 134, 145, 113, 140, 148, 132, 131, 105, 133]
(50000, 17)
10
torch.Size([2196, 100]) torch.Size([47804, 100]) torch.Size([2196, 1]) torch.Size([47804, 1])
high biased ratio:  0.04392


### Step 3. pruning neurons most associated with the Biased group and fine-tuning the model on the Reliable group to produce debiased propensity scores for downstream tasks (ATE estimation).

In [9]:
 
#
trained =  model.state_dict()    
ratio = 0.5
 
### Pruning on PS model
model_p = M_pruned(in_N=d , m= n_hidden, depth= n_depth)   
model_p.load_state_dict(trained)

#Test
model_p.eval()   
with torch.no_grad():   
    test_outputs, p_list  = model_p.pre_act(Xt  )   
    test_outputs = test_outputs.squeeze()
    predicted_prob = test_outputs.numpy() 
    actual_treatment = At.numpy().squeeze()
pred = (predicted_prob >=0.5)
p_list_l = [item[~con] for item in p_list]
p_list_h = [item[con] for item in p_list]

#L2 for each neuron
epsilon = 1e-12
sl = []
for i in range(len(p_list_l)):
    norm1_l =  torch.mean( p_list_l[i]**2 , dim=0)
    norm1_h =  torch.mean( p_list_h[i]**2 , dim=0)
    score = torch.div(norm1_h, norm1_l+epsilon)
    sl.append(score)

sm = torch.cat([i.unsqueeze(1) for i in sl], dim=1) #score matrix 

num_neuron = n_hidden   
num_to_prune = int(num_neuron* n_depth * ratio)
values, indices = torch.topk(sm.flatten(), num_to_prune)
row_indices = indices // sm.size(1)   
col_indices = indices % sm.size(1)  

state_p = model_p.state_dict()
with torch.no_grad():
    state_p['stack.'+str(0)+'.weight'][row_indices[col_indices==0],:]=0
    state_p['stack.'+str(0)+'.bias'][row_indices[col_indices==0]]=0
    state_p['stack.'+str(1)+'.weight'][row_indices[col_indices==1],:]=0
    state_p['stack.'+str(1)+'.weight'][:,row_indices[col_indices==0]]=0
    state_p['stack.'+str(1)+'.bias'][row_indices[col_indices==1]]=0
    state_p['stack.'+str(2)+'.weight'][row_indices[col_indices==2],:]=0
    state_p['stack.'+str(2)+'.weight'][:,row_indices[col_indices==1]]=0
    state_p['stack.'+str(2)+'.bias'][row_indices[col_indices==2]]=0 
    state_p['stack.'+str(3)+'.weight'][:,row_indices[col_indices==2]]=0
    model_p.load_state_dict(state_p)


### finetuning on low biased data after pruning
criterion = nn.BCELoss()  
optimizer = optim.Adam(model_p.parameters(), lr=0.0001, weight_decay=5e-6 )
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)    

n_epochs = 3
batch_size = 80   
#
model_p.eval()   
with torch.no_grad():   
    test_outputs = model_p(X_l).squeeze()  
    predicted_prob = test_outputs.numpy()
    
actual_treatment = A_l.numpy().squeeze()
pred = (predicted_prob >=0.5)
print('initialzation', np.sum(pred==actual_treatment )/pred.shape[0],optimizer.param_groups[-1]['lr'])
#
for epoch in range(n_epochs):  
    model_p.train()  
    running_loss = 0.0
    permutation = torch.randperm(X_l.size(0)) 
    for i in range(0, X_l.shape[0], batch_size):

        ind = permutation[i:i + batch_size]
        batch_x, batch_a = X_l[ind], A_l[ind]
        out = model_p(batch_x)  
        optimizer.zero_grad()
        loss = criterion(out,batch_a)
        loss.backward()
        # Manually prevent weight updates for frozen neurons
        with torch.no_grad():
            model_p.stack[0].weight.grad[row_indices[col_indices==0],:]=0
            model_p.stack[0].bias.grad[row_indices[col_indices==0]]=0
            model_p.stack[1].weight.grad[row_indices[col_indices==1],:]=0
            model_p.stack[1].weight.grad[:,row_indices[col_indices==0]]=0
            model_p.stack[1].bias.grad[row_indices[col_indices==1]]=0
            model_p.stack[2].weight.grad[row_indices[col_indices==2],:]=0
            model_p.stack[2].weight.grad[:,row_indices[col_indices==1]]=0
            model_p.stack[2].bias.grad[row_indices[col_indices==2]]=0
            model_p.stack[3].weight.grad[:,row_indices[col_indices==2]]=0
        #--------------------
        optimizer.step()
        running_loss += loss.item()
    scheduler.step()

    model_p.eval()   
    with torch.no_grad():   
        test_outputs = model_p(Xt).squeeze()  
        predicted_prob = test_outputs.numpy()
    actual_treatment = At.numpy().squeeze()
    pred = (predicted_prob >=0.5) 
        
    if (epoch + 1) %1 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}, Loss: {running_loss/len(Xt)*batch_size}', np.sum(pred==actual_treatment )/pred.shape[0],optimizer.param_groups[-1]['lr'])
 
 
 
 
model_p.eval()   
with torch.no_grad():
    ps_p = model_p(Xt).numpy().squeeze()

# Compute trimming thresholds
lower = np.percentile(ps_p, 1)
upper = np.percentile(ps_p, 99)
keep_mask = (ps_p >= lower) & (ps_p <= upper)
A_trimmed = A[keep_mask]
Y_trimmed = Y[keep_mask]
ps_p_trimmed = ps_p[keep_mask]

w = np.ones(ps_p_trimmed.shape[0])
w[A_trimmed==1] = 1/ps_p_trimmed[A_trimmed==1]
w[A_trimmed==0] = 1/(1-ps_p_trimmed[A_trimmed==0])
ipwlogrr_after, ipwselogrr_after = RR(w, A_trimmed,Y_trimmed) 
print(  'UNO IPW ATE: ',  np.exp(ipwlogrr_after) )
    

#bootstrap
n_bootstrap = 100 
boot_ates_p = []
for _ in range(n_bootstrap):
    # Resample data with replacement
    sample_indices = resample(range(len(df)), replace=True)
         
    # Compute trimming thresholds
    lower = np.percentile(ps_p[sample_indices], 1)
    upper = np.percentile(ps_p[sample_indices], 99)
    keep_mask = (ps_p[sample_indices] >= lower) & (ps_p[sample_indices] <= upper)
        
    A_trimmed = A[sample_indices][keep_mask]
    Y_trimmed = Y[sample_indices][keep_mask]
    ps_trimmed = ps_p[sample_indices][keep_mask]

    w = np.ones(ps_trimmed.shape[0])
    w[A_trimmed==1] = 1/ps_trimmed[A_trimmed==1]
    w[A_trimmed==0] = 1/(1-ps_trimmed[A_trimmed==0])
    boot_ate_p,_  = RR(w, A_trimmed,Y_trimmed) 
    boot_ates_p.append(boot_ate_p)

# Compute standard error (SE) 
ate_w_se_p = np.std(boot_ates_p, ddof=1)   
print(  'UNO Boot ATE SE: ',   ate_w_se_p ) 
print( 'UNO  HR and CI: ',  np.exp(ipwlogrr_after), np.exp(ipwlogrr_after -1.96*ate_w_se_p)   , np.exp(ipwlogrr_after + 1.96*ate_w_se_p))


initialzation 0.5932767132457535 0.0001
Epoch 1/3, Loss: 0.6477119615554809 0.59298 0.0001
Epoch 2/3, Loss: 0.6426017105102539 0.59298 0.0001
Epoch 3/3, Loss: 0.6323227171897888 0.59298 0.0001
UNO IPW ATE:  0.9976995313213807
UNO Boot ATE SE:  0.011331630311012779
UNO  HR and CI:  0.9976995313213807 0.97578489216559 1.0201063398202146
